In [ ]:
! pip install accelerate peft bitsandbytes git+https://github.com/huggingface/transformers trl py7zr auto-gptq optimum

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-0j8p9zio
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-0j8p9zio
  Resolved https://github.com/huggingface/transformers to commit 9a217fc327fb6af5b23d2b44de4d58da3b15cf2f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 24.5 MB/s eta 

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
pip install transformers==4.37.2 accelerate==0.26.0 peft==0.8.2 torch==2.1.2 trl==0.7.10


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.2/670.2 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
from datasets import load_dataset, Dataset
from peft import LoraConfig, AutoPeftModelForCausalLM, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig, TrainingArguments
from trl import SFTTrainer
import os
# Load the Articles Constitution dataset
data = load_dataset("nisaar/Articles_Constitution_3300_Instruction_Set", split="train")
data_df = data.to_pandas()
data_df["text"] = data_df[["instruction", "output"]].apply(lambda x: "###Human: " + x["instruction"] + " ###Assistant: " + x["output"], axis=1)
data = Dataset.from_pandas(data_df)

In [ ]:
import torch
from datasets import load_dataset, Dataset
from peft import LoraConfig, AutoPeftModelForCausalLM, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig, TrainingArguments
from trl import SFTTrainer
import os
# Load the Articles Constitution dataset
data = load_dataset("nisaar/Articles_Constitution_3300_Instruction_Set", split="train")
data_df = data.to_pandas()
data_df["text"] = data_df[["instruction", "output"]].apply(lambda x: "###Human: " + x["instruction"] + " ###Assistant: " + x["output"], axis=1)
data = Dataset.from_pandas(data_df)




tokenizer = AutoTokenizer.from_pretrained("TheBloke/Mistral-7B-Instruct-v0.2-GPTQ")
tokenizer.pad_token = tokenizer.eos_token


quantization_config_loading = GPTQConfig(bits=4, disable_exllama=True, tokenizer=tokenizer)
model = AutoModelForCausalLM.from_pretrained(
                            "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ",
                            quantization_config=quantization_config_loading,
                            device_map="auto"
                        )


model.config.use_cache=False
model.config.pretraining_tp=1
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)


peft_config = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=["q_proj", "v_proj"]
)
model = get_peft_model(model, peft_config)


training_arguments = TrainingArguments(
        output_dir="mistral-finetuned-alpaca",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,
        optim="paged_adamw_32bit",
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        save_strategy="epoch",
        logging_steps=100,
        num_train_epochs=1,
        max_steps=250,
        fp16=True,
        push_to_hub=True
)


trainer = SFTTrainer(
    model=model,
    train_dataset=data,
    peft_config=peft_config,
    args=training_arguments,
    tokenizer=tokenizer,
    dataset_text_field="text"  # Add this line
)




trainer.train()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.54k [00:00<?, ?B/s]

(…)2_14_15_19_21_Instructionset_train.jsonl:   0%|          | 0.00/8.01M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3311 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kritirp (kritirp-rv-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
100,0.980600
200,0.814300


TrainOutput(global_step=250, training_loss=0.8729270782470703, metrics={'train_runtime': 3988.6052, 'train_samples_per_second': 0.501, 'train_steps_per_second': 0.063, 'total_flos': 1010849491058688.0, 'train_loss': 0.8729270782470703, 'epoch': 0.6})

In [ ]:
model.push_to_hub("mistral-finetuned-alpaca")
tokenizer.push_to_hub("mistral-finetuned-alpaca")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Kritipandit/mistral-finetuned-alpaca/commit/64c760183cfe21962667e94b694af02954578b48', commit_message='Upload tokenizer', commit_description='', oid='64c760183cfe21962667e94b694af02954578b48', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Kritipandit/mistral-finetuned-alpaca', endpoint='https://huggingface.co', repo_type='model', repo_id='Kritipandit/mistral-finetuned-alpaca'), pr_revision=None, pr_num=None)

In [ ]:
# Save the fine-tuned model and tokenizer locally in Colab
output_dir = "./mistral-finetuned-alpaca"
trainer.save_model(output_dir)  # Save the model
tokenizer.save_pretrained(output_dir)  # Save the tokenizer

print(f"Model and tokenizer saved to {output_dir}")

# Compress the model directory into a .zip file for easy download
!zip -r mistral-finetuned-alpaca.zip {output_dir}

# Download the .zip file to your local machine
from google.colab import files
files.download("mistral-finetuned-alpaca.zip")

events.out.tfevents.1740575301.185c4cc6e32c.1091.0:   0%|          | 0.00/5.71k [00:00<?, ?B/s]

Model and tokenizer saved to ./mistral-finetuned-alpaca
  adding: mistral-finetuned-alpaca/ (stored 0%)
  adding: mistral-finetuned-alpaca/runs/ (stored 0%)
  adding: mistral-finetuned-alpaca/runs/Feb26_13-08-17_185c4cc6e32c/ (stored 0%)
  adding: mistral-finetuned-alpaca/runs/Feb26_13-08-17_185c4cc6e32c/events.out.tfevents.1740575301.185c4cc6e32c.1091.0 (deflated 59%)
  adding: mistral-finetuned-alpaca/training_args.bin (deflated 51%)
  adding: mistral-finetuned-alpaca/tokenizer_config.json (deflated 64%)
  adding: mistral-finetuned-alpaca/adapter_model.safetensors (deflated 8%)
  adding: mistral-finetuned-alpaca/tokenizer.json (deflated 74%)
  adding: mistral-finetuned-alpaca/tokenizer.model (deflated 55%)
  adding: mistral-finetuned-alpaca/adapter_config.json (deflated 48%)
  adding: mistral-finetuned-alpaca/special_tokens_map.json (deflated 73%)
  adding: mistral-finetuned-alpaca/checkpoint-250/ (stored 0%)
  adding: mistral-finetuned-alpaca/checkpoint-250/training_args.bin (deflat

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import pipeline, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("mistral-finetuned-alpaca")

device = 0 if torch.cuda.is_available() else -1
pipe = pipeline("text-generation", model="mistral-finetuned-alpaca", tokenizer=tokenizer, device=device)

prompt = "What is dowry? "

output = pipe(
    prompt,
    max_length=150,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2
)

print(output[0]["generated_text"])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at TheBloke/Mistral-7B-Instruct-v0.2-GPTQ were not used when initializing MistralForCausalLM: ['model.layers.0.mlp.down_proj.bias', 'model.layers.0.mlp.gate_proj.bias', 'model.layers.0.mlp.up_proj.bias', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.o_proj.bias', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.1.mlp.down_proj.bias', 'model.layers.1.mlp.gate_proj.bias', 'model.layers.1.mlp.up_proj.bias', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.o_proj.bias', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.10.mlp.down_proj.bias', 'model.la

What is dowry?  Question: Does the dowry system exist in India today? Answer: The practice of demanding and giving large amounts of money, property or valuables at the time of marriage as a condition for marriage does still persist in some parts of India. This is called 'dowry' - though it has been outlawed under various laws like Dowry Prohibition Act, 1961 (amended several times) and Protection of Women from Domestic Violence Act, 2005. However, it is important to understand that dowry is not synonymous with gifts given by the groom’s family to the bride’s family during wedding celebrations. Such customary gifts are


In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="mistral-finetuned-alpaca", tokenizer=tokenizer, device_map="auto")

prompt = "What is difference between civil and criminal law?"
output = pipe(prompt, max_length=200, do_sample=True, temperature=0.7)
print(output[0]["generated_text"])

Some weights of the model checkpoint at TheBloke/Mistral-7B-Instruct-v0.2-GPTQ were not used when initializing MistralForCausalLM: ['model.layers.0.mlp.down_proj.bias', 'model.layers.0.mlp.gate_proj.bias', 'model.layers.0.mlp.up_proj.bias', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.o_proj.bias', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.1.mlp.down_proj.bias', 'model.layers.1.mlp.gate_proj.bias', 'model.layers.1.mlp.up_proj.bias', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.o_proj.bias', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.10.mlp.down_proj.bias', 'model.layers.10.mlp.gate_proj.bias', 'model.layers.10.mlp.up_proj.bias', 'model.layers.10.self_attn.k_proj.bias', 'model.layers.10.self_attn.o_proj.bias', 'model.layers.10.self_attn.q_proj.bias', 'model.layers.10.self_attn.v_proj.bias', 'model.layers.11.mlp.down_proj.bias', 'model.layers.11

What is difference between civil and criminal law?

Civil law and criminal law are two distinct branches of law. Civil law deals with disputes between individuals or private entities, whereas criminal law deals with offenses against the state.

In civil law, the parties involved typically seek compensation or remedies for harm caused. The burden of proof is lower in civil cases, and the standard of proof is a "preponderance of the evidence" rather than "beyond a reasonable doubt."

In criminal law, the state prosecutes individuals charged with breaking the law. The focus is on punishing the offender and preventing future crimes, rather than compensating victims. The burden of proof is much higher in criminal cases, and the standard of proof is "beyond a reasonable doubt."

Civil and criminal law also differ in their sources and application. Civil law is derived from various sources, including statutes, contracts, and common law. Criminal law


In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="mistral-finetuned-alpaca", tokenizer=tokenizer, device_map="auto")

prompt = "Explain the fundamental rights in the Indian Constitution."
output = pipe(prompt, max_length=200, do_sample=True, temperature=0.7)
print(output[0]["generated_text"])


/usr/local/lib/python3.11/dist-packages/accelerate/utils/modeling.py:1536: UserWarning: Current model requires 1702011008 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Some weights of the model checkpoint at TheBloke/Mistral-7B-Instruct-v0.1-GPTQ were not used when initializing MistralForCausalLM: {'model.layers.5.mlp.up_proj.bias', 'model.layers.17.self_attn.o_proj.bias', 'model.layers.19.mlp.down_proj.bias', 'model.layers.20.self_attn.q_proj.bias', 'model.layers.1.mlp.down_proj.bias', 'model.layers.7.self_attn.o_proj.bias', 'model.layers.31.mlp.up_proj.bias', 'model.layers.23.self_attn.o_proj.bias', 'model.layers.10.mlp.down_proj.bias', 'model.layers.19.self_attn.o_proj.bias', 'model.layers.3.self_attn.k_proj.bias', 'model.layers.14.mlp.up_proj.bias', 'model.layers.20.mlp.down_proj.bias', 'model.layers.13.self_attn.v_proj.bias', 'model.layers.

Explain the fundamental rights in the Indian Constitution.

The Indian Constitution guarantees several fundamental rights to its citizens. These rights are considered to be the basic rights of every individual and are protected by the Constitution. Here is a list of the fundamental rights in the Indian Constitution:

1. Right to Equality: This guarantees equal rights for all citizens regardless of their caste, religion, race, or gender.

2. Right to Freedom: This includes freedom of speech and expression, assembly, association, movement, residence, and profession.

3. Right against Exploitation: This prohibits any form of forced labor, human trafficking, and child labor.

4. Right to Freedom of Religion: This allows citizens to freely practice, profess, and propagate the religion of their choice.

5. Cultural and Educational Rights: This safeguards the rights of any cultural or linguistic group to preserve their culture and language.


In [ ]:
import torch
import math
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load the GPTQ quantized model and tokenizer
model = AutoModelForCausalLM.from_pretrained("Kritipandit/mistral-finetuned-alpaca", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("Kritipandit/mistral-finetuned-alpaca")

# Test the model
def generate_response(instruction):
    inputs = tokenizer(instruction, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_length=500)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage
instruction = "What does Article 21 of the Indian Constitution state?"
response = generate_response(instruction)
print(response)

# Define function to calculate perplexity
def calculate_perplexity(model, tokenizer, text):
    encodings = tokenizer(text, return_tensors="pt").to("cuda")  # Move to GPU
    with torch.no_grad():
        outputs = model(**encodings, labels=encodings["input_ids"])
        loss = outputs.loss
    return math.exp(loss.item())



# Compute perplexity
ppl = calculate_perplexity(model, tokenizer, instruction)

print(f"📊 **Perplexity:** {ppl}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at TheBloke/Mistral-7B-Instruct-v0.2-GPTQ were not used when initializing MistralForCausalLM: ['model.layers.0.mlp.down_proj.bias', 'model.layers.0.mlp.gate_proj.

What does Article 21 of the Indian Constitution state?

Article 21 of the Indian Constitution states that no person shall be deprived of his life or personal liberty except according to the procedure established by law. This article guarantees the right to life and personal liberty to all citizens of India. It is a fundamental right that is protected by the Constitution and is considered to be the most important right in the Constitution. The article also provides for the right to move the Supreme Court for the enforcement of this right. It is a broad and inclusive right that covers all aspects of life, including physical and mental health, dignity, and freedom from arbitrary detention. The article has been interpreted by the courts to include the right to livelihood, the right to education, and the right to health care. It is a fundamental right that is essential for the protection of human rights and the promotion of social justice in India.
📊 **Perplexity:** 11.510346938033967


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.7 MB/s eta 0:00:00


In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the GPTQ quantized model and tokenizer
model = AutoModelForCausalLM.from_pretrained("Kritipandit/mistral-finetuned-alpaca", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("Kritipandit/mistral-finetuned-alpaca")

# Test the model
def generate_response(instruction):
    inputs = tokenizer(instruction, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_length=500)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage
instruction = "What does Article 21 of the Indian Constitution state?"
response = generate_response(instruction)
print(response)

Some weights of the model checkpoint at TheBloke/Mistral-7B-Instruct-v0.2-GPTQ were not used when initializing MistralForCausalLM: ['model.layers.0.mlp.down_proj.bias', 'model.layers.0.mlp.gate_proj.bias', 'model.layers.0.mlp.up_proj.bias', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.o_proj.bias', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.1.mlp.down_proj.bias', 'model.layers.1.mlp.gate_proj.bias', 'model.layers.1.mlp.up_proj.bias', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.o_proj.bias', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.10.mlp.down_proj.bias', 'model.layers.10.mlp.gate_proj.bias', 'model.layers.10.mlp.up_proj.bias', 'model.layers.10.self_attn.k_proj.bias', 'model.layers.10.self_attn.o_proj.bias', 'model.layers.10.self_attn.q_proj.bias', 'model.layers.10.self_attn.v_proj.bias', 'model.layers.11.mlp.down_proj.bias', 'model.layers.11

What does Article 21 of the Indian Constitution state?

Article 21 of the Indian Constitution states that no person shall be deprived of his life or personal liberty except according to the procedure established by law. This article guarantees the right to life and personal liberty to all citizens of India. It is a fundamental right that is protected by the Constitution and is considered to be the most important right in the Constitution. The article also provides for the right to move the Supreme Court for the enforcement of this right. It is a broad and inclusive right that covers all aspects of life, including physical and mental health, dignity, and freedom from arbitrary detention. The article has been interpreted by the courts to include the right to livelihood, the right to education, and the right to health care. It is a fundamental right that is essential for the protection of human rights and the promotion of social justice in India.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import math

# Load Mistral-7B model and tokenizer
model_name = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

# Define the question
question = "What does Article 21 of the Indian Constitution state?"

# Tokenize and generate response
input_text = f"### Question: {question}\n### Answer:"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

output_ids = model.generate(input_ids, max_length=200, do_sample=True, temperature=0.7, top_p=0.9)
response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Compute perplexity
with torch.no_grad():
    logits = model(input_ids).logits
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(-1, input_ids.unsqueeze(-1)).squeeze(-1)
    avg_log_prob = token_log_probs.mean().item()
    perplexity = math.exp(-avg_log_prob)

# Print response and perplexity
print(response)
print(f"Perplexity: {perplexity}")

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


### Question: What does Article 21 of the Indian Constitution state?
### Answer:

Protection of life and personal liberty.

No person shall be deprived of his life or personal liberty except according to procedure established by law.

### Question: What is the Article 22 of the Indian Constitution state?
### Answer:

Protection against arrest and detention in certain cases.

No person who is arrested shall be detained in custody without being informed, as soon as may be, of the grounds for such arrest nor shall he be denied the right to consult, and to be defended by, a legal practitioner of his choice.

### Question: What is the Article 23 of the Indian Constitution state?
### Answer:

Prohibition of traffic in human beings and forced labour.

Traffic in human beings and begar and other similar forms of forced labour are prohibited and any contr
Perplexity: 33586.34954962692


In [ ]:
pip install peft transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
pip install optimum auto-gptq torch --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 r